# Buyer Segmentation and Investment Profiling for Real Estate Market Intelligence

## Exploratory Data Analysis (EDA)

This notebook performs a comprehensive exploratory analysis of the processed buyer and property datasets to understand demographic patterns, investment behavior, financing preferences, geographic trends, and property portfolio characteristics.

The analysis will support feature engineering and clustering model development.

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', None)

# Load processed dataset
buyer_master = pd.read_csv('../data/processed/buyer_master_dataset.csv')

print('Buyer master dataset shape:', buyer_master.shape)
buyer_master.head()

Buyer master dataset shape: (10000, 22)


,listing_id,tower_number,transaction_date,unit_category,unit_number,floor_area_sqft,sale_price,listing_status,client_ref,client_id,client_type,first_name,last_name,date_of_birth,gender,country,region,acquisition_purpose,satisfaction_score,loan_applied,referral_channel,age
0,1012,1,01-01-2024,Apartment,12,1160.36,"$300,385.62",Sold,C0027,C0027,Individual,Grant,Weber,1968-09-08,M,Usa,California,Home,5.0,Yes,Website,58.0
1,1015,1,01-01-2024,Apartment,15,782.25,"$208,930.81",Sold,C0097,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1021,1,01-01-2024,Apartment,21,756.21,"$218,585.92",Sold,C0113,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1030,1,01-01-2024,Apartment,30,743.09,"$246,172.68",Sold,C0141,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2016,2,01-01-2024,Apartment,16,701.66,"$212,265.67",Sold,C0146,C0146,Individual,Hazel,Ayers,1963-02-01,M,Usa,Colorado,Home,2.0,No,Website,63.0


In [5]:
buyer_master.dtypes

listing_id               int64
tower_number             int64
transaction_date           str
unit_category              str
unit_number              int64
floor_area_sqft        float64
sale_price                 str
listing_status             str
client_ref                 str
client_id                  str
client_type                str
first_name                 str
last_name                  str
date_of_birth              str
gender                     str
country                    str
region                     str
acquisition_purpose        str
satisfaction_score     float64
loan_applied               str
referral_channel           str
age                    float64
dtype: object

In [50]:
# Convert numeric columns safely
numeric_cols = ['sale_price', 'floor_area_sqft', 'age', 'satisfaction_score']
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

buyer_master = pd.read_csv('../data/processed/buyer_master_dataset.csv')

# Clean sale_price
buyer_master['sale_price'] = (
    buyer_master['sale_price']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.replace('₹', '', regex=False)
    .str.replace('$', '', regex=False)
    .str.strip()
)

buyer_master['sale_price'] = pd.to_numeric(
    buyer_master['sale_price'],
    errors='coerce'
)

# Clean floor area
buyer_master['floor_area_sqft'] = pd.to_numeric(
    buyer_master['floor_area_sqft'],
    errors='coerce'
)

# Clean satisfaction score
buyer_master['satisfaction_score'] = pd.to_numeric(
    buyer_master['satisfaction_score'],
    errors='coerce'
)

# Clean age
buyer_master['age'] = pd.to_numeric(
    buyer_master['age'],
    errors='coerce'
)

print(buyer_master[['sale_price', 'floor_area_sqft', 'satisfaction_score', 'age']].dtypes)

print(buyer_master[['sale_price', 'floor_area_sqft']].describe())
for col in numeric_cols:
    buyer_master[col] = pd.to_numeric(buyer_master[col], errors='coerce')

# Check data types
buyer_master[numeric_cols].dtypes

sale_price            float64
floor_area_sqft       float64
satisfaction_score    float64
age                   float64
dtype: object
          sale_price  floor_area_sqft
count   10000.000000     10000.000000
mean   344374.678683      1139.941412
std    131563.099417       418.373967
min     97402.800000       410.710000
25%    232173.642500       782.200000
50%    330680.845000      1110.880000
75%    448852.575000      1499.000000
max    736652.270000      1957.160000


sale_price            float64
floor_area_sqft       float64
age                   float64
satisfaction_score    float64
dtype: object

In [51]:
buyer_profile = buyer_master.groupby('client_id').agg(
    total_properties_owned=('listing_id', 'count'),
    total_investment_value=('sale_price', 'sum'),
    average_property_value=('sale_price', 'mean'),
    average_floor_area=('floor_area_sqft', 'mean'),
    age=('age', 'first'),
    client_type=('client_type', 'first'),
    gender=('gender', 'first'),
    country=('country', 'first'),
    region=('region', 'first'),
    acquisition_purpose=('acquisition_purpose', 'first'),
    loan_applied=('loan_applied', 'first'),
    referral_channel=('referral_channel', 'first'),
    satisfaction_score=('satisfaction_score', 'mean')
).reset_index()

buyer_profile.to_csv('../data/processed/buyer_profile_dataset.csv', index=False)

print('Buyer profile dataset shape:', buyer_profile.shape)
buyer_profile.head()

Buyer profile dataset shape: (855, 14)


,client_id,total_properties_owned,total_investment_value,average_property_value,average_floor_area,age,client_type,gender,country,region,acquisition_purpose,loan_applied,referral_channel,satisfaction_score
0,C0001,4,1246764.72,311691.180,983.885,58.0,Individual,F,Usa,California,Home,Yes,Website,4.0
1,C0003,5,1661457.59,332291.518,1058.110,67.0,Individual,M,Usa,California,Home,Yes,Agency,4.0
2,C0006,5,1514131.06,302826.212,989.590,69.0,Individual,M,Usa,California,Home,Yes,Website,3.0
3,C0009,5,1820832.35,364166.470,1305.644,51.0,Individual,M,Usa,California,Investment,No,Agency,5.0
4,C0013,5,2142750.89,428550.178,1407.376,64.0,Individual,M,Usa,California,Home,Yes,Website,1.0


In [52]:
# Make a copy
buyer_profile = buyer_profile.copy()

# Clean sale_price related features if they contain commas or currency symbols
money_cols = ['total_investment_value', 'average_property_value']

for col in money_cols:
    buyer_profile[col] = (
        buyer_profile[col]
        .astype(str)
        .str.replace(',', '', regex=False)
        .str.replace('₹', '', regex=False)
        .str.replace('$', '', regex=False)
        .str.strip()
    )
    buyer_profile[col] = pd.to_numeric(buyer_profile[col], errors='coerce')

# Convert remaining numeric columns
numeric_cols = [
    'age',
    'satisfaction_score',
    'total_properties_owned',
    'total_investment_value',
    'average_property_value',
    'average_floor_area'
]

for col in numeric_cols:
    buyer_profile[col] = pd.to_numeric(buyer_profile[col], errors='coerce')

# Check missing values
missing = buyer_profile[numeric_cols].isnull().sum().to_frame(name='Missing Values')
missing['Percentage'] = (missing['Missing Values'] / len(buyer_profile)) * 100
display(missing)

# Fill missing values
for col in numeric_cols:
    buyer_profile[col] = buyer_profile[col].fillna(buyer_profile[col].median())

# Verify
buyer_profile[numeric_cols].describe()

,Missing Values,Percentage
age,0,0.0
satisfaction_score,0,0.0
total_properties_owned,0,0.0
total_investment_value,0,0.0
average_property_value,0,0.0
average_floor_area,0,0.0


,age,satisfaction_score,total_properties_owned,total_investment_value,average_property_value,average_floor_area
count,855.000000,855.000000,855.000000,8.550000e+02,855.000000,855.000000
mean,55.274854,3.032749,3.646784,1.266353e+06,349668.099199,1154.010203
std,17.206774,1.391292,0.806392,3.412942e+05,70892.117146,225.084877
min,26.000000,1.000000,3.000000,4.636120e+05,154537.316667,564.010000
25%,41.000000,2.000000,3.000000,1.028993e+06,295125.972083,984.800000
50%,56.000000,3.000000,4.000000,1.221443e+06,344285.486667,1137.294286
75%,69.000000,4.000000,4.000000,1.444481e+06,394858.306250,1303.625000
max,93.000000,5.000000,9.000000,3.082362e+06,547651.480000,1754.936667


In [53]:
kpis = {
    'Total Buyers': buyer_profile.shape[0],
    'Total Transactions': buyer_master.shape[0],
    'Countries': buyer_profile['country'].nunique(),
    'Regions': buyer_profile['region'].nunique(),
    'Average Age': round(buyer_profile['age'].mean(), 1),
    'Average Satisfaction': round(buyer_profile['satisfaction_score'].mean(), 2),
    'Average Investment Value': round(buyer_profile['total_investment_value'].mean(), 2),
    'Median Investment Value': round(buyer_profile['total_investment_value'].median(), 2),
    'Average Properties Owned': round(buyer_profile['total_properties_owned'].mean(), 2)
}

pd.DataFrame(list(kpis.items()), columns=['Metric', 'Value'])

,Metric,Value
0,Total Buyers,855.00
1,Total Transactions,10000.00
2,Countries,10.00
3,Regions,54.00
4,Average Age,55.30
5,Average Satisfaction,3.03
6,Average Investment Value,1266353.43
7,Median Investment Value,1221442.92
8,Average Properties Owned,3.65


In [54]:
fig = px.histogram(
    buyer_profile,
    x='age',
    nbins=20,
    marginal='violin',
    title='Age Distribution of Buyers'
)

fig.update_layout(template='plotly_white')
fig.show()

In [55]:
gender_counts = buyer_profile['gender'].value_counts().reset_index()
gender_counts.columns = ['Gender', 'Count']

fig = px.bar(
    gender_counts,
    x='Gender',
    y='Count',
    color='Gender',
    text='Count',
    title='Buyer Gender Distribution'
)

fig.update_layout(template='plotly_white')
fig.show()

In [56]:
fig = px.violin(
    buyer_profile,
    x='client_type',
    y='total_investment_value',
    color='client_type',
    box=True,
    points='outliers',
    title='Investment Value Distribution by Client Type'
)

fig.update_layout(template='plotly_white')
fig.show()

In [57]:
fig = px.violin(
    buyer_profile,
    x='acquisition_purpose',
    y='total_investment_value',
    color='acquisition_purpose',
    box=True,
    points='outliers',
    title='Investment Value by Acquisition Purpose'
)

fig.update_layout(template='plotly_white')
fig.show()

In [58]:
top_countries = buyer_profile['country'].value_counts().head(10).reset_index()
top_countries.columns = ['Country', 'Buyers']

fig = px.bar(
    top_countries,
    x='Country',
    y='Buyers',
    color='Buyers',
    text='Buyers',
    title='Top 10 Countries by Buyer Count'
)

fig.update_layout(template='plotly_white')
fig.show()

In [59]:
fig = px.histogram(
    buyer_profile,
    x='total_properties_owned',
    nbins=15,
    marginal='box',
    title='Properties Owned per Buyer'
)

fig.update_layout(template='plotly_white')
fig.show()

In [60]:
fig = px.histogram(
    buyer_profile,
    x='total_investment_value',
    nbins=30,
    marginal='violin',
    title='Total Investment Value Distribution'
)

fig.update_layout(template='plotly_white')
fig.show()

In [61]:
fig = px.violin(
    buyer_profile,
    x='client_type',
    y='satisfaction_score',
    color='client_type',
    box=True,
    points='outliers',
    title='Satisfaction Score by Client Type'
)

fig.update_layout(template='plotly_white')
fig.show()

In [62]:
corr = buyer_profile[numeric_cols].corr()

fig = px.imshow(
    corr,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    title='Correlation Matrix of Buyer Features'
)

fig.update_layout(template='plotly_white')
fig.show()